# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio



# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9478, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 9478 (delta 44), reused 45 (delta 31), pack-reused 9327 (from 2)
Receiving objects: 100% (9478/9478), 2.78 MiB | 7.76 MiB/s, done.
Resolving deltas: 100% (6829/6829), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 84.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB

In [2]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 154 (delta 84), reused 107 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 941.65 KiB | 22.42 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/Titans_jax


In [5]:
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm

# Our custom Titans integration
from gemma_titans import Gemma3_1B_Titans, TitansLayerCache
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 90% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


## 1. Initialization & 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

In [ ]:
# Define the model configuration
gemma_config = gm.nn.Gemma3_1B.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model.titans_layer_indices = [11] # Add Titans memory only to the middle layer

# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 8
max_length = 256

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"
    
    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()

def get_train_dataset():
    return kd.data.py.Tfds(
        name="squad",
        split="train[:100]",
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        num_workers=4,
        transforms=[
            format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="input",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["input", "target", "loss_mask"]),
        ],
    )

# Configure Trainer
trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    train_ds=get_train_dataset(),
    model=model,

    # --- THE MAGIC: SkipTitans ---
    # Loads base weights from checkpoint, leaves Titans layers as random init
    init_transform=SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir')
    ),

    num_train_steps=10000,
    train_losses={
        "xentropy": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },

    # --- PREVENT TPU OOM & REDUCE COMPILATION SPIKE ---
    # Using optax.apply_every for Gradient Accumulation (Optax 0.2.6)
    optimizer=optax.apply_every(4)(
        kd.optim.partial_updates(
            optax.adam(learning_rate=1e-4),
            mask=kd.optim.select(["memory", "memory_gate"]),
        )
    ),
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    sharding=kd.sharding.ShardingStrategy(),
)

print("Trainer initialized. Starting compilation and training...")

# Uncomment to run the actual training loop:
state, aux = trainer.train()

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # Extract only the keys that belong to Titans (memory, memory_gate)
    original_params, titans_params = titans_tree_utils.split_titans_params(state.params)

    checkpointer = ocp.StandardCheckpointer()
    # force=True is deprecated, wait_until_finished makes sure the background thread completes
    checkpointer.save(os.path.abspath(save_dir), args=ocp.args.StandardSave(titans_params))
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_dir}")

def load_titans_weights(state: kd.train.TrainState, load_dir: str) -> kd.train.TrainState:
    checkpointer = ocp.StandardCheckpointer()
    titans_params = checkpointer.restore(os.path.abspath(load_dir), args=ocp.args.StandardRestore(state.params))

    # Split the current state, swap out the titans portion, and merge back
    original_params, _ = titans_tree_utils.split_titans_params(state.params)
    merged_params = titans_tree_utils.merge_titans_params(original_params, titans_params)

    return state.replace(params=merged_params)

# Usage:
# save_titans_weights(state, "./saved_titans_delta")
# state = load_titans_weights(state, "./saved_titans_delta")

## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)

    logits = output.logits
    new_cache = output.cache

    # 2. Logic to extract the response token goes here
    return logits, new_cache

# Initialize empty state
print("Initializing Cache...")
# cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
# user_tokens = jnp.array([[1, 2, 3, 4]])
# logits, cache = dialogue_turn(state.params, cache, user_tokens)
# print("Memory state updated automatically in cache.")